# Qwen3-8B QLoRA Fine-Tuning — Step 1: Pronoun Resolution (v5)

**T4-optimized config**: rsLoRA on T4×2 via DDP + inference post-processing

| Setting | Value | Rationale |
|---|---|---|
| Adapter | **LoRA + rsLoRA** | DoRA OOMs on T4 |
| Rank / Alpha | **16 / 32** | More capacity for precise entity tracking |
| LoRA dropout | **0** | Enables full Unsloth fast-patching; dropout hurts near-copy fidelity |
| NEFTune | **Disabled** | Near-copy task — embedding noise hurts faithful reproduction |
| Learning rate | **5e-6** | Reduced from 2e-5 — prevents loss collapse in <1 epoch |
| Scheduler | Cosine + 10% warmup | Gentler decay than linear |
| Weight decay | 0.05 | L2 regularization |
| Early stopping | patience=5 on eval_loss | Increased patience for slower LR |
| Epochs | 8 (max) | Early stopping will cut shorter |
| Batch size | 2/GPU × 2 grad_accum = 8 effective | |
| Multi-GPU | DDP via torchrun | Uses both T4 GPUs |
| Data | **Oversampled: no-change 3x, low-change 2x, high-change 3x** | Aggressive no-change to fix 21% hallucination |
| Inference | **Greedy decoding** (do_sample=False) | Eliminates sampling noise for near-copy task |
| **Post-processing** | **Word-level input anchoring** | Copy unchanged spans verbatim from input; keep only model’s deliberate changes |
| Metrics | **RAW vs PP side-by-side + F1=1.0 diagnostic** | Full 50-sample comparison saved to eval_results.txt |

## 1. Install Dependencies

In [1]:
%%capture
!pip install unsloth
!pip install --no-deps trl sft-trainer peft accelerate bitsandbytes triton xformers

In [2]:
import torch, os

NUM_GPUS = torch.cuda.device_count()
for i in range(NUM_GPUS):
    name = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    bf16 = torch.cuda.is_bf16_supported()
    print(f"GPU {i}: {name} | {vram:.1f} GB | bf16={bf16}")
print(f"\nTotal GPUs: {NUM_GPUS}")
print(f"Training mode: {'DDP (torchrun)' if NUM_GPUS > 1 else 'Single GPU'}")

GPU 0: Tesla T4 | 15.6 GB | bf16=True
GPU 1: Tesla T4 | 15.6 GB | bf16=True

Total GPUs: 2
Training mode: DDP (torchrun)


## 2. Write Training Script

DDP-compatible training with rsLoRA, data rebalancing, and token accuracy tracking.

In [3]:
%%writefile /kaggle/working/train_step1.py
#!/usr/bin/env python
"""Step 1: Pronoun Resolution — DDP training with rsLoRA (v4)."""
import os, sys, json, time, glob, shutil, logging, random, difflib
import torch

# Suppress noisy logs before any HF imports
for _ln in ("httpx", "urllib3", "huggingface_hub",
            "transformers.tokenization_utils_base"):
    logging.getLogger(_ln).setLevel(logging.WARNING)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

logging.basicConfig(format="%(asctime)s [%(levelname)s] %(message)s", level=logging.INFO)
logger = logging.getLogger(__name__)

LOCAL_RANK = int(os.environ.get("LOCAL_RANK", 0))
WORLD_SIZE = int(os.environ.get("WORLD_SIZE", 1))
IS_MAIN = LOCAL_RANK == 0

# ===================== CONFIGURATION =====================
MAX_SEQ_LENGTH = 2048
MODEL_NAME = "unsloth/Qwen3-8B-unsloth-bnb-4bit"

TRAIN_FILE = "/kaggle/input/datasets/mohammadasadolahi/medical-relation-extraction-training/step1_pronoun_train.jsonl"
EVAL_FILE = "/kaggle/input/datasets/mohammadasadolahi/medical-relation-extraction-training/step1_pronoun_eval.jsonl"

OUTPUT_DIR = "/kaggle/working/step1_training"
ADAPTER_DIR = "/kaggle/working/step1_pronoun_adapter"
CHECKPOINT_INPUT = "/kaggle/input/qwen3-8b-training-checkpoints"

LORA_R = 16
LORA_ALPHA = 32
USE_RSLORA = True
LORA_DROPOUT = 0.0

NUM_EPOCHS = 8
BATCH_SIZE = 2
GRAD_ACCUM = 2
LR = 5e-6
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.05
SAVE_STEPS = 50
EVAL_STEPS = 50
LOGGING_STEPS = 10
EARLY_STOPPING_PATIENCE = 5
SEED = 3407
# =========================================================


def rebalance_training_data(train_file):
    """Oversample examples based on pronoun change count.

    no change  -> 3x   (duplicate twice) — strong "don't edit" signal
    1-3 changes -> 2x   (duplicate once)
    4+ changes  -> 3x   (duplicate twice)
    """
    with open(train_file) as f:
        samples = [json.loads(line) for line in f]

    extra_indices = []
    n_no_change = n_low = n_high = 0
    no_change_indices = []

    for i, sample in enumerate(samples):
        convos = sample["conversations"]
        user_text = convos[1]["content"]
        asst_text = convos[2]["content"]

        inp_words = user_text.split()
        out_words = asst_text.split()
        sm = difflib.SequenceMatcher(None, inp_words, out_words)
        n_ops = sum(1 for tag, *_ in sm.get_opcodes() if tag != "equal")

        if n_ops == 0:
            n_no_change += 1
            no_change_indices.append(i)
            extra_indices.extend([i, i])
        elif n_ops <= 3:
            n_low += 1
            extra_indices.append(i)
        else:
            n_high += 1
            extra_indices.extend([i, i])

    stats = {
        "no_change": n_no_change,
        "no_change_multiplier": "3x",
        "low_change_2x": n_low,
        "high_change_3x": n_high,
        "original_size": len(samples),
        "rebalanced_size": len(samples) + len(extra_indices),
    }
    return extra_indices, stats


def main():
    if IS_MAIN:
        logger.info("=" * 70)
        logger.info("STEP 1: Pronoun Resolution Training (v4)")
        logger.info("=" * 70)
        logger.info(f"Config: r={LORA_R} alpha={LORA_ALPHA} lr={LR} dropout={LORA_DROPOUT} wd={WEIGHT_DECAY}")
        logger.info(f"GPUs={WORLD_SIZE} batch={BATCH_SIZE}x{GRAD_ACCUM}x{WORLD_SIZE}={BATCH_SIZE*GRAD_ACCUM*WORLD_SIZE} epochs={NUM_EPOCHS} patience={EARLY_STOPPING_PATIENCE}")

    # ---- Data rebalancing ----
    extra_indices, rebalance_stats = rebalance_training_data(TRAIN_FILE)
    if IS_MAIN:
        logger.info(f"Rebalancing: {rebalance_stats['no_change']} unchanged (3x), "
                     f"{rebalance_stats['low_change_2x']} low-change (2x), "
                     f"{rebalance_stats['high_change_3x']} high-change (3x)")
        logger.info(f"Dataset: {rebalance_stats['original_size']} -> {rebalance_stats['rebalanced_size']} samples")

    # ---- Resume checkpoint ----
    resume_checkpoint = None
    if os.path.exists(CHECKPOINT_INPUT):
        checkpoints = sorted(glob.glob(f"{CHECKPOINT_INPUT}/checkpoint-*"))
        if checkpoints:
            latest_src = checkpoints[-1]
            dest = os.path.join(OUTPUT_DIR, os.path.basename(latest_src))
            if IS_MAIN:
                os.makedirs(OUTPUT_DIR, exist_ok=True)
                if not os.path.exists(dest):
                    shutil.copytree(latest_src, dest)
                logger.info(f"RESUMING from: {dest}")
            if WORLD_SIZE > 1:
                torch.distributed.barrier()
            resume_checkpoint = dest
    elif IS_MAIN:
        logger.info("Training from scratch")

    # ---- Load model ----
    from unsloth import FastModel

    model, tokenizer = FastModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        load_in_8bit=False,
        full_finetuning=False,
    )

    model = FastModel.get_peft_model(
        model,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_rslora=USE_RSLORA,
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )

    if IS_MAIN:
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in model.parameters())
        logger.info(f"LoRA rsLoRA={USE_RSLORA}: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

    # ---- Dataset ----
    from datasets import load_dataset
    from unsloth.chat_templates import get_chat_template, standardize_sharegpt

    tokenizer = get_chat_template(tokenizer, chat_template="chatml")

    dataset = load_dataset("json", data_files={"train": TRAIN_FILE, "test": EVAL_FILE})

    # Apply rebalancing via index duplication
    all_indices = list(range(len(dataset["train"]))) + extra_indices
    random.seed(SEED)
    random.shuffle(all_indices)
    dataset["train"] = dataset["train"].select(all_indices)

    dataset["train"] = standardize_sharegpt(dataset["train"])
    dataset["test"] = standardize_sharegpt(dataset["test"])

    def formatting_func(examples):
        convos = examples["conversations"]
        texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False) for c in convos]
        return {"text": texts}

    dataset = dataset.map(formatting_func, batched=True, remove_columns=dataset["train"].column_names)

    if IS_MAIN:
        logger.info(f"Train: {len(dataset['train'])} samples | Eval: {len(dataset['test'])} samples")
        lens = []
        for text in dataset["train"]["text"]:
            toks = tokenizer(text, return_length=True, truncation=False)
            lens.append(toks["length"][0])
        lens.sort()
        n = len(lens)
        logger.info(f"Token lengths — min:{lens[0]} med:{lens[n//2]} p95:{lens[int(n*0.95)]} max:{lens[-1]}")
        over = sum(1 for l in lens if l > MAX_SEQ_LENGTH)
        if over:
            logger.warning(f"{over}/{n} exceed max_seq_length={MAX_SEQ_LENGTH}")

    # ---- Trainer ----
    from trl import SFTTrainer, SFTConfig
    from unsloth import train_on_responses_only
    from transformers import EarlyStoppingCallback

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    sft_config = SFTConfig(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=WARMUP_RATIO,
        optim="adamw_8bit",
        weight_decay=WEIGHT_DECAY,
        fp16=True,
        bf16=False,
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,
        dataset_text_field="text",
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        logging_steps=LOGGING_STEPS,
        logging_first_step=True,
        report_to="none",
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        seed=SEED,
        max_grad_norm=1.0,
        ddp_find_unused_parameters=False,
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset["train"],
        eval_dataset=dataset["test"],
        args=sft_config,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )

    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )

    if IS_MAIN:
        batch = next(iter(trainer.get_train_dataloader()))
        labels = batch["labels"][0]
        masked = (labels == -100).sum().item()
        total_labels = labels.numel()
        active = total_labels - masked
        logger.info(f"Label masking: {masked} masked, {active} active ({100*active/total_labels:.1f}% response tokens)")
        if active == 0:
            logger.error("ALL LABELS MASKED — aborting!")
            sys.exit(1)

    if IS_MAIN:
        logger.info(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB | Starting training...")

    t0 = time.time()
    train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)
    elapsed = time.time() - t0

    # ---- Summary ----
    if IS_MAIN:
        best_metric = getattr(trainer.state, "best_metric", None)
        best_ckpt = getattr(trainer.state, "best_model_checkpoint", "?")

        logger.info("=" * 70)
        logger.info("TRAINING COMPLETE")
        logger.info("=" * 70)
        logger.info(f"  Config:     r={LORA_R} alpha={LORA_ALPHA} lr={LR} dropout={LORA_DROPOUT}")
        logger.info(f"  Dataset:    {rebalance_stats['original_size']} raw -> {rebalance_stats['rebalanced_size']} rebalanced / {len(dataset['test'])} eval")
        logger.info(f"  Stopped at: epoch {train_result.metrics.get('epoch', 0):.2f} (step {int(trainer.state.global_step)})")
        logger.info(f"  Train loss: {train_result.metrics.get('train_loss', 0):.6f}")
        if best_metric is not None:
            logger.info(f"  Best eval:  loss={best_metric:.6f} (at {best_ckpt})")
        logger.info(f"  Time:       {elapsed:.0f}s ({elapsed/60:.1f} min)")

    if WORLD_SIZE > 1:
        torch.distributed.barrier()

    if IS_MAIN:
        os.makedirs(ADAPTER_DIR, exist_ok=True)
        model.save_pretrained(ADAPTER_DIR)
        tokenizer.save_pretrained(ADAPTER_DIR)

        # Save system prompt for inference
        with open(TRAIN_FILE) as f:
            first_sample = json.loads(f.readline())
        with open(f"{ADAPTER_DIR}/system_prompt.txt", "w") as f:
            f.write(first_sample["conversations"][0]["content"])

        summary = {
            "model": MODEL_NAME,
            "lora_r": LORA_R,
            "lora_alpha": LORA_ALPHA,
            "use_rslora": USE_RSLORA,
            "lora_dropout": LORA_DROPOUT,
            "learning_rate": LR,
            "weight_decay": WEIGHT_DECAY,
            "neftune": None,
            "effective_batch_size": BATCH_SIZE * GRAD_ACCUM * WORLD_SIZE,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "rebalance_stats": rebalance_stats,
            "training_time_seconds": elapsed,
            "train_metrics": train_result.metrics,
            "eval_metrics": {"eval_loss": best_metric} if best_metric else {},
        }
        with open(f"{ADAPTER_DIR}/training_summary.json", "w") as f:
            json.dump(summary, f, indent=2, default=str)

        logger.info(f"  Adapter saved: {ADAPTER_DIR}")


if __name__ == "__main__":
    main()


Writing /kaggle/working/train_step1.py


## 3. Resume Check

If `qwen3-8b-training-checkpoints` dataset is attached, copies checkpoint for resume.

In [4]:
import os, glob, shutil

CHECKPOINT_INPUT = "/kaggle/input/qwen3-8b-training-checkpoints"
OUTPUT_DIR = "/kaggle/working/step1_training"

if os.path.exists(CHECKPOINT_INPUT):
    contents = os.listdir(CHECKPOINT_INPUT)
    checkpoints = sorted(glob.glob(f"{CHECKPOINT_INPUT}/checkpoint-*"))
    print(f"Checkpoint dataset attached. Contents: {contents}")
    if checkpoints:
        print(f"Found {len(checkpoints)} checkpoint(s). Latest: {checkpoints[-1]}")
        print("Training script will auto-resume from this checkpoint.")
    else:
        print("No checkpoint-* dirs found. Training from scratch.")
else:
    print("No checkpoint dataset attached. Training from scratch.")
    print("To resume: attach 'qwen3-8b-training-checkpoints' dataset to this notebook.")

No checkpoint dataset attached. Training from scratch.
To resume: attach 'qwen3-8b-training-checkpoints' dataset to this notebook.


## 4. Launch Training

In [5]:
import torch

num_gpus = torch.cuda.device_count()

if num_gpus > 1:
    print(f"Launching DDP training on {num_gpus} GPUs...")
    !torchrun --nproc_per_node={num_gpus} /kaggle/working/train_step1.py
else:
    print("Launching single-GPU training...")
    !python /kaggle/working/train_step1.py

Launching DDP training on 2 GPUs...
W0518 05:05:47.603000 72 torch/distributed/run.py:852] 
W0518 05:05:47.603000 72 torch/distributed/run.py:852] *****************************************
W0518 05:05:47.603000 72 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0518 05:05:47.603000 72 torch/distributed/run.py:852] *****************************************
2026-05-18 05:05:49,396 [INFO] ======================================================================
2026-05-18 05:05:49,396 [INFO] STEP 1: Pronoun Resolution Training (v4)
2026-05-18 05:05:49,396 [INFO] ======================================================================
2026-05-18 05:05:49,397 [INFO] Config: r=16 alpha=32 lr=5e-06 dropout=0.0 wd=0.05
2026-05-18 05:05:49,397 [INFO] GPUs=2 batch=2x2x2=8 epochs=8 patience=5
2026-05-18 05:05:49

## 5. Results & Pronoun Resolution Evaluation

In [6]:
import json, os

ADAPTER_DIR = "/kaggle/working/step1_pronoun_adapter"
summary_path = f"{ADAPTER_DIR}/training_summary.json"

if os.path.exists(summary_path):
    with open(summary_path) as f:
        s = json.load(f)
    rb = s.get("rebalance_stats", {})
    print("=" * 70)
    print("TRAINING SUMMARY")
    print("=" * 70)
    print(f"  Model:      {s['model']}")
    print(f"  LoRA:       r={s['lora_r']} alpha={s['lora_alpha']} rslora={s['use_rslora']} dropout={s['lora_dropout']}")
    print(f"  Training:   lr={s['learning_rate']} wd={s['weight_decay']} neftune={s.get('neftune') or 'off'}")
    print(f"  Data:       {rb.get('original_size', '?')} raw -> {rb.get('rebalanced_size', '?')} rebalanced")
    print(f"  Batch:      {s['effective_batch_size']}")
    print(f"  Train loss: {s['train_metrics'].get('train_loss', 'N/A')}")
    print(f"  Eval loss:  {s['eval_metrics'].get('eval_loss', 'N/A')}")
    print(f"  Time:       {s['training_time_seconds']:.0f}s ({s['training_time_seconds']/60:.1f} min)")
    print("=" * 70)
else:
    print("No training summary found. Training may have been interrupted.")

TRAINING SUMMARY
  Model:      unsloth/Qwen3-8B-unsloth-bnb-4bit
  LoRA:       r=16 alpha=32 rslora=True dropout=0.0
  Training:   lr=5e-06 wd=0.05 neftune=off
  Data:       1385 raw -> 3452 rebalanced
  Batch:      8
  Train loss: 0.03650595860047774
  Eval loss:  0.013157389126718044
  Time:       17725s (295.4 min)


### Pronoun Resolution Metrics (v5 — with post-processing)

Generates predictions for all 50 eval samples using **greedy decoding**, then applies
**word-level post-processing**: unchanged spans are copied verbatim from the input text,
while the model’s deliberate changes (pronoun resolutions) are preserved.

This targets the “F1=1.0 but not EXACT” fidelity problem identified in v4 analysis:
the model finds correct pronoun positions but introduces micro-mutations to surrounding text.

**Evaluation shows both RAW and POST-PROCESSED (PP) metrics side-by-side** to measure
the exact impact of post-processing.

**Includes diagnostic section** for F1=1.0-but-not-EXACT samples showing exact word-level
differences between reference and prediction to pinpoint whether issues are in equal spans
or replacement content.

Full per-sample results (INPUT/EXPECTED/RAW/PP text) saved to `/kaggle/working/eval_results.txt`.

In [7]:
import json, re, difflib, os, collections
import torch

ADAPTER_DIR = "/kaggle/working/step1_pronoun_adapter"
EVAL_CANDIDATES = [
    "/kaggle/input/datasets/mohammadasadolahi/medical-relation-extraction-training/step1_pronoun_eval.jsonl",
    "/kaggle/input/medical-relation-extraction-training/step1_pronoun_eval.jsonl",
]
RESULTS_FILE = "/kaggle/working/eval_results.txt"


def find_eval_file():
    for p in EVAL_CANDIDATES:
        if os.path.exists(p):
            return p
    return None


def postprocess_prediction(input_text, raw_prediction):
    """Copy word-level 'equal' spans verbatim from input, keep model's changes.

    Fixes text fidelity: unchanged regions become byte-identical to input.
    """
    inp_w = input_text.split()
    pred_w = raw_prediction.split()

    result = []
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(None, inp_w, pred_w).get_opcodes():
        if tag == "equal":
            result.extend(inp_w[i1:i2])
        elif tag != "delete":
            result.extend(pred_w[j1:j2])

    return " ".join(result)


def compute_pronoun_metrics(input_text, reference, prediction):
    """Word-level change detection metrics for pronoun resolution."""
    inp_w = input_text.split()
    ref_w = reference.split()
    pred_w = prediction.split()

    ref_changes = set()
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(None, inp_w, ref_w).get_opcodes():
        if tag != "equal":
            for i in range(i1, i2):
                ref_changes.add(i)

    pred_changes = set()
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(None, inp_w, pred_w).get_opcodes():
        if tag != "equal":
            for i in range(i1, i2):
                pred_changes.add(i)

    tp = len(ref_changes & pred_changes)
    fp = len(pred_changes - ref_changes)
    fn = len(ref_changes - pred_changes)

    precision = tp / (tp + fp) if (tp + fp) > 0 else (1.0 if fn == 0 else 0.0)
    recall = tp / (tp + fn) if (tp + fn) > 0 else 1.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    char_sim = difflib.SequenceMatcher(None, reference, prediction).ratio()
    exact = reference.strip() == prediction.strip()
    is_no_change = input_text.strip() == reference.strip()
    hallucinated = is_no_change and not exact

    return {
        "exact_match": exact,
        "char_similarity": char_sim,
        "n_ref_changes": len(ref_changes),
        "n_pred_changes": len(pred_changes),
        "tp": tp, "fp": fp, "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "is_no_change": is_no_change,
        "hallucinated": hallucinated,
    }


def categorize_sample(m):
    """Classify a sample into one of 7 diagnostic categories."""
    if m["is_no_change"]:
        if m["exact_match"]:
            return "EXACT (no-change)"
        else:
            return "HALLUCINATION"
    else:
        if m["exact_match"]:
            return "EXACT"
        elif m["tp"] == 0 and m["fp"] == 0:
            return "MISSED ALL"
        elif m["tp"] > 0 and m["fp"] == 0:
            return "PARTIAL (conservative)"
        elif m["tp"] > 0 and m["fp"] > 0:
            return "PARTIAL (mixed)"
        else:
            return "OVER-EDITED"


if not os.path.exists(ADAPTER_DIR):
    print("Adapter not found — skipping evaluation.")
else:
    EVAL_FILE = find_eval_file()
    if EVAL_FILE is None:
        print("ERROR: Cannot find step1_pronoun_eval.jsonl")
    else:
        import logging
        for name in ("httpx", "urllib3", "huggingface_hub"):
            logging.getLogger(name).setLevel(logging.WARNING)

        from unsloth import FastModel
        from unsloth.chat_templates import get_chat_template

        model, tokenizer = FastModel.from_pretrained(
            model_name=ADAPTER_DIR, max_seq_length=2048, load_in_4bit=True,
        )
        tokenizer = get_chat_template(tokenizer, chat_template="chatml")
        FastModel.for_inference(model)

        with open(f"{ADAPTER_DIR}/system_prompt.txt") as f:
            SYSTEM_PROMPT = f.read().strip()

        eval_samples = []
        with open(EVAL_FILE) as f:
            for line in f:
                eval_samples.append(json.loads(line))

        print(f"Evaluating {len(eval_samples)} samples: greedy decoding + post-processing\n")

        raw_metrics = []
        pp_metrics = []
        all_responses = []

        for i, sample in enumerate(eval_samples):
            user_text = sample["conversations"][1]["content"]
            expected = sample["conversations"][2]["content"]

            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_text},
            ]
            inputs = tokenizer.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=True,
                return_tensors="pt", enable_thinking=False,
            ).to(model.device)

            with torch.no_grad():
                outputs = model.generate(
                    input_ids=inputs, max_new_tokens=1536,
                    do_sample=False,
                )
            response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
            if "<think>" in response:
                response = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL).strip()

            pp_response = postprocess_prediction(user_text, response)

            m_raw = compute_pronoun_metrics(user_text, expected, response)
            m_raw["category"] = categorize_sample(m_raw)
            raw_metrics.append(m_raw)

            m_pp = compute_pronoun_metrics(user_text, expected, pp_response)
            m_pp["category"] = categorize_sample(m_pp)
            pp_metrics.append(m_pp)

            all_responses.append({
                "input": user_text,
                "expected": expected,
                "raw": response,
                "pp": pp_response,
            })

            if (i + 1) % 10 == 0:
                print(f"  ... {i+1}/{len(eval_samples)} done")

        n = len(raw_metrics)

        # ============================================================
        # RAW per-sample table
        # ============================================================
        print(f"\n{'='*110}")
        print(f"PER-SAMPLE RESULTS — RAW ({n} samples)")
        print(f"{'='*110}")
        hdr = f"{'#':>3} | {'Category':<24} | {'Sim':>5} | {'P':>5} | {'R':>5} | {'F1':>5} | {'tp':>3} | {'fp':>3} | {'fn':>3} | {'RefChg':>6} | {'PredChg':>7}"
        print(hdr)
        print("-" * len(hdr))
        for i, m in enumerate(raw_metrics):
            print(f"{i+1:>3} | {m['category']:<24} | {m['char_similarity']:.3f} | {m['precision']:.3f} | {m['recall']:.3f} | {m['f1']:.3f} | {m['tp']:>3} | {m['fp']:>3} | {m['fn']:>3} | {m['n_ref_changes']:>6} | {m['n_pred_changes']:>7}")

        # ============================================================
        # PP per-sample table
        # ============================================================
        print(f"\n{'='*110}")
        print(f"PER-SAMPLE RESULTS — POST-PROCESSED ({n} samples)")
        print(f"{'='*110}")
        print(hdr)
        print("-" * len(hdr))
        for i, m in enumerate(pp_metrics):
            print(f"{i+1:>3} | {m['category']:<24} | {m['char_similarity']:.3f} | {m['precision']:.3f} | {m['recall']:.3f} | {m['f1']:.3f} | {m['tp']:>3} | {m['fp']:>3} | {m['fn']:>3} | {m['n_ref_changes']:>6} | {m['n_pred_changes']:>7}")

        # ============================================================
        # Side-by-side: which samples changed?
        # ============================================================
        print(f"\n{'='*100}")
        print("POST-PROCESSING IMPACT — samples where metrics changed")
        print(f"{'='*100}")
        chdr = f"{'#':>3} | {'Cat RAW':<24} | {'Cat PP':<24} | {'Sim_R':>5} | {'Sim_PP':>6} | {'dSim':>6} | {'Ex_R':>4} | {'Ex_PP':>5}"
        print(chdr)
        print("-" * len(chdr))
        n_improved = 0
        n_exact_gained = 0
        sim_deltas = []
        for i in range(n):
            mr, mp = raw_metrics[i], pp_metrics[i]
            sim_d = mp["char_similarity"] - mr["char_similarity"]
            if abs(sim_d) > 1e-6 or mr["exact_match"] != mp["exact_match"]:
                n_improved += 1
                if not mr["exact_match"] and mp["exact_match"]:
                    n_exact_gained += 1
                sim_deltas.append(sim_d)
                ex_r = "Y" if mr["exact_match"] else "N"
                ex_pp = "Y" if mp["exact_match"] else "N"
                print(f"{i+1:>3} | {mr['category']:<24} | {mp['category']:<24} | {mr['char_similarity']:.3f} | {mp['char_similarity']:.4f} | {sim_d:+.4f} | {ex_r:>4} | {ex_pp:>5}")
        if n_improved == 0:
            print("  (no samples changed)")
        else:
            avg_delta = sum(sim_deltas) / len(sim_deltas)
            print(f"\n  Samples affected: {n_improved}/{n}")
            print(f"  New exact matches gained: {n_exact_gained}")
            print(f"  Avg similarity delta (affected): {avg_delta:+.4f}")

        # ============================================================
        # Category summaries side-by-side
        # ============================================================
        cats = ["EXACT (no-change)", "EXACT", "PARTIAL (conservative)",
                "PARTIAL (mixed)", "OVER-EDITED", "MISSED ALL", "HALLUCINATION"]
        raw_cats = collections.Counter(m["category"] for m in raw_metrics)
        pp_cats = collections.Counter(m["category"] for m in pp_metrics)

        print(f"\n{'='*70}")
        print("CATEGORY SUMMARY — RAW vs POST-PROCESSED")
        print(f"{'='*70}")
        print(f"  {'Category':<24} {'RAW':>5} {'PP':>5} {'Delta':>6}")
        print(f"  {'-'*44}")
        for cat in cats:
            rc = raw_cats.get(cat, 0)
            pc = pp_cats.get(cat, 0)
            d = pc - rc
            ds = f"{d:+d}" if d != 0 else "="
            print(f"  {cat:<24} {rc:>5} {pc:>5} {ds:>6}")

        # ============================================================
        # Aggregate metrics: RAW vs PP
        # ============================================================
        def agg(metrics_list):
            nm = len(metrics_list)
            em = sum(m["exact_match"] for m in metrics_list)
            si = sum(m["char_similarity"] for m in metrics_list) / nm
            pr = sum(m["precision"] for m in metrics_list) / nm
            re_ = sum(m["recall"] for m in metrics_list) / nm
            f1 = sum(m["f1"] for m in metrics_list) / nm
            nc = [m for m in metrics_list if m["is_no_change"]]
            hc = [m for m in metrics_list if not m["is_no_change"]]
            nh = sum(m["hallucinated"] for m in nc) if nc else 0
            hc_em = sum(m["exact_match"] for m in hc) if hc else 0
            hc_r = sum(m["recall"] for m in hc) / len(hc) if hc else 0
            hc_f1 = sum(m["f1"] for m in hc) / len(hc) if hc else 0
            nc_pres = sum(m["exact_match"] for m in nc) if nc else 0
            return {
                "exact": em, "sim": si, "prec": pr, "rec": re_, "f1": f1,
                "halluc": nh, "n_nc": len(nc), "n_hc": len(hc),
                "hc_exact": hc_em, "hc_rec": hc_r, "hc_f1": hc_f1,
                "nc_pres": nc_pres,
            }

        ra = agg(raw_metrics)
        pa = agg(pp_metrics)

        print(f"\n{'='*70}")
        print(f"AGGREGATE METRICS ({n} samples)")
        print(f"{'='*70}")
        print(f"  {'Metric':<25} {'RAW':>10} {'PP':>10} {'Delta':>10}")
        print(f"  {'-'*55}")
        rows = [
            ("Exact match", f"{ra['exact']}/{n} ({100*ra['exact']/n:.1f}%)",
             f"{pa['exact']}/{n} ({100*pa['exact']/n:.1f}%)",
             f"{pa['exact']-ra['exact']:+d}"),
            ("Char similarity", f"{ra['sim']:.4f}", f"{pa['sim']:.4f}",
             f"{pa['sim']-ra['sim']:+.4f}"),
            ("Change precision", f"{ra['prec']:.4f}", f"{pa['prec']:.4f}",
             f"{pa['prec']-ra['prec']:+.4f}"),
            ("Change recall", f"{ra['rec']:.4f}", f"{pa['rec']:.4f}",
             f"{pa['rec']-ra['rec']:+.4f}"),
            ("Change F1", f"{ra['f1']:.4f}", f"{pa['f1']:.4f}",
             f"{pa['f1']-ra['f1']:+.4f}"),
        ]
        for label, rv, pv, dv in rows:
            print(f"  {label:<25} {rv:>10} {pv:>10} {dv:>10}")

        print(f"\n  Samples WITH changes ({ra['n_hc']}):")
        print(f"    Exact:  {ra['hc_exact']} -> {pa['hc_exact']} ({pa['hc_exact']-ra['hc_exact']:+d})")
        print(f"    Recall: {ra['hc_rec']:.4f} -> {pa['hc_rec']:.4f}")
        print(f"    F1:     {ra['hc_f1']:.4f} -> {pa['hc_f1']:.4f}")

        print(f"\n  No-change samples ({ra['n_nc']}):")
        print(f"    Preserved:    {ra['nc_pres']}/{ra['n_nc']} -> {pa['nc_pres']}/{pa['n_nc']}")
        print(f"    Hallucinated: {ra['halluc']}/{ra['n_nc']} -> {pa['halluc']}/{pa['n_nc']}")

        # ============================================================
        # Verdicts: RAW and PP
        # ============================================================
        for label, a in [("RAW", ra), ("POST-PROCESSED", pa)]:
            print(f"\n{'='*70}")
            print(f"VERDICT — {label}")
            print(f"{'='*70}")
            checks = [
                ("Change F1 >= 0.80", a["f1"] >= 0.80),
                ("Exact match >= 50%", a["exact"] / n >= 0.50),
                ("Char similarity >= 0.95", a["sim"] >= 0.95),
            ]
            if a["n_nc"] > 0:
                checks.append(("Hallucination <= 10%",
                                a["halluc"] / a["n_nc"] <= 0.10))
            if a["n_hc"] > 0:
                checks.append(("Change recall >= 0.75", a["hc_rec"] >= 0.75))

            all_pass = True
            for check_label, passed in checks:
                status = "PASS" if passed else "FAIL"
                print(f"  [{status}] {check_label}")
                if not passed:
                    all_pass = False
            result = "ALL CHECKS PASSED" if all_pass else "SOME CHECKS FAILED"
            print(f"\n  >>> {result}")

        # ============================================================
        # Diagnostic: F1=1.0 but not EXACT — WHERE do ref vs pred differ?
        # ============================================================
        print(f"\n{'='*70}")
        print("DIAGNOSTIC: F1=1.0 but not EXACT (word-level diff: reference vs prediction)")
        print(f"{'='*70}")
        print("Shows exact word differences to identify if issue is in replacement")
        print("content (wrong entity name) or text fidelity (whitespace/punctuation).\n")
        diag_count = 0
        for i, (mr, mp, r) in enumerate(zip(raw_metrics, pp_metrics, all_responses)):
            if mr["f1"] >= 0.999 and not mr["exact_match"]:
                diag_count += 1
                ref_w = r["expected"].split()
                raw_w = r["raw"].split()
                pp_w = r["pp"].split()
                print(f"  Sample {i+1} | raw_sim={mr['char_similarity']:.3f} pp_sim={mp['char_similarity']:.3f} | pp_exact={'Y' if mp['exact_match'] else 'N'}")

                diffs_raw = []
                for tag, a1, a2, b1, b2 in difflib.SequenceMatcher(None, ref_w, raw_w).get_opcodes():
                    if tag != "equal":
                        rp = " ".join(ref_w[a1:a2]) if a1 < a2 else "(none)"
                        pp_ = " ".join(raw_w[b1:b2]) if b1 < b2 else "(none)"
                        diffs_raw.append(f"    {tag}: ref='{rp}' raw='{pp_}'")
                diffs_pp = []
                for tag, a1, a2, b1, b2 in difflib.SequenceMatcher(None, ref_w, pp_w).get_opcodes():
                    if tag != "equal":
                        rp = " ".join(ref_w[a1:a2]) if a1 < a2 else "(none)"
                        pp_ = " ".join(pp_w[b1:b2]) if b1 < b2 else "(none)"
                        diffs_pp.append(f"    {tag}: ref='{rp}' pp='{pp_}'")

                if diffs_raw:
                    print(f"    RAW diffs ({len(diffs_raw)}):")
                    for d in diffs_raw[:8]:
                        print(d)
                if diffs_pp:
                    print(f"    PP diffs ({len(diffs_pp)}):")
                    for d in diffs_pp[:8]:
                        print(d)
                if not diffs_pp:
                    print("    PP: no diffs (EXACT!)")
                print()

        if diag_count == 0:
            print("  (no F1=1.0 non-EXACT samples found)")

        # ============================================================
        # Diagnostic: catastrophic outliers (sim < 0.80)
        # ============================================================
        print(f"\n{'='*70}")
        print("DIAGNOSTIC: Catastrophic outliers (raw sim < 0.80)")
        print(f"{'='*70}")
        for i, (mr, mp, r) in enumerate(zip(raw_metrics, pp_metrics, all_responses)):
            if mr["char_similarity"] < 0.80:
                print(f"\n  Sample {i+1} | raw_sim={mr['char_similarity']:.3f} pp_sim={mp['char_similarity']:.3f}")
                print(f"    Category: {mr['category']} -> {mp['category']}")
                print(f"    tp={mr['tp']} fp={mr['fp']} fn={mr['fn']}")
                print(f"    Input len:  {len(r['input'].split())} words")
                print(f"    Ref len:    {len(r['expected'].split())} words")
                print(f"    Raw len:    {len(r['raw'].split())} words")
                print(f"    PP len:     {len(r['pp'].split())} words")

        # ============================================================
        # Save full results to file
        # ============================================================
        with open(RESULTS_FILE, "w", encoding="utf-8") as rf:
            rf.write(f"EVAL RESULTS — {n} samples, greedy + post-processing\n")
            rf.write("=" * 80 + "\n\n")
            for i, (mr, mp, r) in enumerate(zip(raw_metrics, pp_metrics, all_responses)):
                rf.write(f"--- Sample {i+1} ---\n")
                rf.write(f"  RAW:  {mr['category']} | sim={mr['char_similarity']:.3f}"
                         f" P={mr['precision']:.3f} R={mr['recall']:.3f}"
                         f" F1={mr['f1']:.3f} tp={mr['tp']} fp={mr['fp']} fn={mr['fn']}\n")
                rf.write(f"  PP:   {mp['category']} | sim={mp['char_similarity']:.3f}"
                         f" P={mp['precision']:.3f} R={mp['recall']:.3f}"
                         f" F1={mp['f1']:.3f} tp={mp['tp']} fp={mp['fp']} fn={mp['fn']}\n")
                rf.write(f"  INPUT:    {r['input'][:400]}\n")
                rf.write(f"  EXPECTED: {r['expected'][:400]}\n")
                rf.write(f"  RAW:      {r['raw'][:400]}\n")
                rf.write(f"  PP:       {r['pp'][:400]}\n\n")
        print(f"\nFull results saved to {RESULTS_FILE}")

        del model
        torch.cuda.empty_cache()


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

unsloth/Qwen3-8B-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth: Will map <|im_end|> to EOS = <|im_end|>.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=1536) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Evaluating 50 samples: greedy decoding + post-processing



...

  ... 30/50 done


Both `max_new_tokens` (=1536) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1536) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1536) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1536) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  ... 40/50 done


Both `max_new_tokens` (=1536) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1536) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1536) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1536) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  ... 50/50 done

PER-SAMPLE RESULTS — RAW (50 samples)
  # | Category                 |   Sim |     P |     R |    F1 |  tp |  fp |  fn | RefChg | PredChg
---------------------------------------------------------------------------------------------------
  1 | HALLUCINATION            | 0.981 | 0.000 | 1.000 | 0.000 |   0 |   1 |   0 |      0 |       1
  2 | PARTIAL (mixed)          | 0.992 | 0.500 | 0.500 | 0.500 |   1 |   1 |   1 |      2 |       2
  3 | PARTIAL (conservative)   | 0.996 | 1.000 | 0.800 | 0.889 |   4 |   0 |   1 |      5 |       4
  4 | EXACT                    | 1.000 | 1.000 | 1.000 | 1.000 |   2 |   0 |   0 |      2 |       2
  5 | PARTIAL (mixed)          | 0.996 | 0.667 | 1.000 | 0.800 |   2 |   1 |   0 |      2 |       3
  6 | HALLUCINATION            | 0.981 | 0.000 | 1.000 | 0.000 |   0 |   3 |   0 |      0 |       3
  7 | EXACT (no-change)        | 1.000 | 1.000 | 1.000 | 1.000 |   0 |   0 |   0 |      0 |       0
  8 | EXACT (no-change)        | 1.000 | 1.0

## 6. Upload Adapter & Checkpoints to Kaggle Datasets

Saves the LoRA adapter and training checkpoints as Kaggle datasets for reuse and resume.

In [8]:
import json, os, shutil, glob, requests

KAGGLE_USERNAME = "mohammadasadolahi"
ADAPTER_DIR = "/kaggle/working/step1_pronoun_adapter"
CHECKPOINT_DIR = "/kaggle/working/step1_training"


def upload_to_kaggle_dataset(folder, dataset_slug, title, version_notes=None):
    """Upload a folder as a Kaggle dataset (create or update)."""
    # Write dataset-metadata.json
    meta = {
        "title": title,
        "id": f"{KAGGLE_USERNAME}/{dataset_slug}",
        "licenses": [{"name": "CC0-1.0"}]
    }
    with open(os.path.join(folder, "dataset-metadata.json"), "w") as f:
        json.dump(meta, f, indent=2)

    # Try Kaggle API (works natively inside Kaggle notebooks)
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate()

        if version_notes:
            result = api.dataset_create_version(
                folder=folder, version_notes=version_notes,
                quiet=False, delete_old_versions=False,
            )
            print(f"  Updated dataset version: {result}")
        else:
            try:
                result = api.dataset_create_new(
                    folder=folder, public=False, quiet=False,
                )
                print(f"  Created new dataset: {result}")
            except Exception:
                result = api.dataset_create_version(
                    folder=folder, version_notes="Update",
                    quiet=False, delete_old_versions=False,
                )
                print(f"  Updated existing dataset: {result}")

        print(f"  URL: https://www.kaggle.com/datasets/{KAGGLE_USERNAME}/{dataset_slug}")
        return True

    except Exception as e:
        print(f"  Kaggle API failed: {e}")
        print(f"  Files are saved at {folder} — download from notebook output.")
        return False


# ---- Upload adapter weights ----
print("=" * 60)
print("1. UPLOADING ADAPTER WEIGHTS")
print("=" * 60)
if os.path.exists(ADAPTER_DIR):
    files = os.listdir(ADAPTER_DIR)
    total_mb = sum(
        os.path.getsize(os.path.join(ADAPTER_DIR, f))
        for f in files if os.path.isfile(os.path.join(ADAPTER_DIR, f))
    ) / 1e6
    print(f"  Adapter: {len(files)} files, {total_mb:.1f} MB")
    upload_to_kaggle_dataset(
        ADAPTER_DIR,
        "qwen3-8b-step1-pronoun-adapter",
        "Qwen3-8B Step1 Pronoun LoRA Adapter",
    )
else:
    print("  No adapter found. Training may have been interrupted.")

# ---- Upload latest checkpoint for resume ----
print(f"\n{'='*60}")
print("2. UPLOADING CHECKPOINT FOR RESUME")
print("=" * 60)
checkpoints = sorted(glob.glob(f"{CHECKPOINT_DIR}/checkpoint-*"))
if checkpoints:
    latest = checkpoints[-1]
    step_num = os.path.basename(latest).split("-")[-1]
    print(f"  Latest checkpoint: {os.path.basename(latest)}")

    # Stage checkpoint for upload
    staging = "/kaggle/working/checkpoint_upload"
    if os.path.exists(staging):
        shutil.rmtree(staging)
    shutil.copytree(latest, os.path.join(staging, os.path.basename(latest)))

    # Also copy trainer_state.json from output_dir if it exists
    trainer_state = os.path.join(CHECKPOINT_DIR, "trainer_state.json")
    if os.path.exists(trainer_state):
        shutil.copy2(trainer_state, staging)

    upload_to_kaggle_dataset(
        staging,
        "qwen3-8b-training-checkpoints",
        "Qwen3-8B Training Checkpoints",
        version_notes=f"Step 1 checkpoint at step {step_num}",
    )
else:
    print("  No checkpoints found.")

1. UPLOADING ADAPTER WEIGHTS
  Adapter: 8 files, 186.1 MB
  Created new dataset: {"ref": null, "url": null, "status": "error", "error": "The requested title \"Qwen3-8B Step1 Pronoun LoRA Adapter\" is already in use by a dataset. Please choose another title.", "invalidTags": []}
  URL: https://www.kaggle.com/datasets/mohammadasadolahi/qwen3-8b-step1-pronoun-adapter

2. UPLOADING CHECKPOINT FOR RESUME
  Latest checkpoint: checkpoint-850
Skipping folder: checkpoint-850; use '--dir-mode' to upload folders
  Kaggle API failed: 403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/CreateDatasetVersion
  Files are saved at /kaggle/working/checkpoint_upload — download from notebook output.


In [9]:
print("\n" + "=" * 60)
print("ALL DONE — Step 1 (Pronoun Resolution)")
print("=" * 60)
print(f"  Adapter:     /kaggle/working/step1_pronoun_adapter/")
print(f"  Checkpoints: /kaggle/working/step1_training/")
print()
print("To RESUME if interrupted:")
print("  1. Attach 'qwen3-8b-training-checkpoints' dataset to this notebook")
print("  2. Re-run all cells — training resumes from last checkpoint")
print()
print("Next step: Update this notebook for Step 2 (Indirect Reference Transfer)")


ALL DONE — Step 1 (Pronoun Resolution)
  Adapter:     /kaggle/working/step1_pronoun_adapter/
  Checkpoints: /kaggle/working/step1_training/

To RESUME if interrupted:
  1. Attach 'qwen3-8b-training-checkpoints' dataset to this notebook
  2. Re-run all cells — training resumes from last checkpoint

Next step: Update this notebook for Step 2 (Indirect Reference Transfer)
